# Longitudinal analysis: Food.com recipe community

This notebook contains longitudinal analyses of the Food.com recipe community. It includes a reviewer-contributor network analysis, an ego-network example, and a separate descriptive analysis of vegan and vegetarian trends.

The first code cell downloads the Food.com dataset with KaggleHub. The reviewer-contributor analysis uses all valid review interactions from 2000 through 2018 and matches each reviewed recipe to its contributor. It does not filter by recipe tags or dietary type. Since the dataset records reviews rather than explicit social ties, a reviewer-contributor edge means that a reviewer reviewed at least one recipe by that contributor; it is not a friendship or follow relationship.

In [1]:
from pathlib import Path

import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("shuyangli94/food-com-recipes-and-user-interactions")
)
print(f"Path to dataset files: {dataset_path}")

interaction_files = list(dataset_path.rglob("RAW_interactions.csv"))
if not interaction_files:
    raise FileNotFoundError(
        f"RAW_interactions.csv was not found under the downloaded dataset path: {dataset_path}"
    )
if len(interaction_files) > 1:
    raise RuntimeError(
        "Expected one RAW_interactions.csv file, found multiple: "
        f"{interaction_files}"
    )

INTERACTIONS_PATH = interaction_files[0]
print(f"Using interactions file: {INTERACTIONS_PATH}")

c:\Users\Julian\.vscode\projects\socialnetwork\social-network-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 267M/267M [00:17<00:00, 16.3MB/s] 

Extracting files...


Path to dataset files: C:\Users\Julian\.cache\kagglehub\datasets\shuyangli94\food-com-recipes-and-user-interactions\versions\2
Using interactions file: C:\Users\Julian\.cache\kagglehub\datasets\shuyangli94\food-com-recipes-and-user-interactions\versions\2\RAW_interactions.csv


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

EGO_USER_ID = None  # Set to a user ID (for example, "1533") to choose a specific ego.

interactions = pd.read_csv(
    INTERACTIONS_PATH,
    dtype={"user_id": "string", "recipe_id": "string"},
)
required_columns = {"user_id", "recipe_id", "date"}
missing_columns = required_columns.difference(interactions.columns)
if missing_columns:
    raise ValueError(f"RAW_interactions.csv is missing columns: {sorted(missing_columns)}")

events = interactions[["user_id", "recipe_id", "date"]].copy()
events["date"] = pd.to_datetime(events["date"], errors="coerce", utc=True)
invalid_dates = int(events["date"].isna().sum())
missing_ids = int(events[["user_id", "recipe_id"]].isna().any(axis=1).sum())
events = events.dropna(subset=["user_id", "recipe_id", "date"])
if events.empty:
    raise ValueError("No rows have valid user_id, recipe_id, and date values.")

events["user_id"] = events["user_id"].astype("string")
events["recipe_id"] = events["recipe_id"].astype("string")
events["year"] = events["date"].dt.year
rows_before_deduplication = len(events)
events = events.drop_duplicates(["year", "user_id", "recipe_id"])
duplicate_events_removed = rows_before_deduplication - len(events)

if EGO_USER_ID is None:
    user_activity = events.groupby("user_id").agg(
        active_years=("year", "nunique"),
        reviewed_recipes=("recipe_id", "nunique"),
    )
    ego_id = str(
        user_activity.sort_values(
            ["active_years", "reviewed_recipes"],
            ascending=[False, False],
            kind="stable",
        ).index[0]
    )
else:
    ego_id = str(EGO_USER_ID)
    if ego_id not in set(events["user_id"]):
        raise ValueError(f"EGO_USER_ID {ego_id!r} does not occur in the valid interactions.")

print(f"Ego user: {ego_id}")
print(f"Rows removed for invalid dates: {invalid_dates}")
print(f"Rows removed for missing user/recipe IDs: {missing_ids}")
print(f"Repeated user-recipe reviews in the same year removed: {duplicate_events_removed}")

Ego user: 8629
Rows removed for invalid dates: 0
Rows removed for missing user/recipe IDs: 0
Repeated user-recipe reviews in the same year removed: 0


## Subquestion 3: Network growth and the emergence of top authorities (2000-2018)

This section uses **all** reviewer interactions, not only vegan or vegetarian recipes. Each reviewed recipe is linked to its `contributor_id`, creating a directed reviewer-to-contributor tie. Annual activity is counted by review date; cumulative network size counts each reviewer, contributor, and reviewer-contributor tie once at its first observed year. An authority is operationalized as a contributor with many distinct reviewers. The final top authorities are ranked by cumulative distinct reviewers through 2018, and their emergence year is the first year they enter the cumulative top ten. These are measures of attention in this dataset, not proof of offline or platform-wide influence; contributor IDs are used because the dataset does not provide verified public names.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

NETWORK_START_YEAR = 2000
NETWORK_END_YEAR = 2018
network_years = list(range(NETWORK_START_YEAR, NETWORK_END_YEAR + 1))
recipes_path = Path(dataset_path) / "RAW_recipes.csv"
if not recipes_path.is_file():
    raise FileNotFoundError(f"RAW_recipes.csv was not found at {recipes_path}")

recipe_contributors = pd.read_csv(
    recipes_path,
    usecols=["id", "contributor_id"],
    dtype={"id": "string", "contributor_id": "string"},
)
if recipe_contributors["id"].isna().any() or recipe_contributors["contributor_id"].isna().any():
    raise ValueError("Recipe IDs and contributor IDs must be present for network attribution.")
if recipe_contributors["id"].duplicated().any():
    raise ValueError("RAW_recipes.csv contains duplicate recipe IDs; contributor mapping is ambiguous.")

network_review_events = events.loc[
    events["year"].between(NETWORK_START_YEAR, NETWORK_END_YEAR),
    ["year", "user_id", "recipe_id"],
].merge(
    recipe_contributors,
    left_on="recipe_id",
    right_on="id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_recipe_count = int(network_review_events["_merge"].ne("both").sum())
if unmatched_recipe_count:
    raise ValueError(
        f"{unmatched_recipe_count} review events could not be matched to a recipe contributor."
    )
network_review_events = network_review_events.drop(columns=["id", "_merge"])
network_review_events = network_review_events.dropna(subset=["contributor_id"])
if network_review_events.empty:
    raise ValueError("No valid reviewer-contributor events were found from 2000 through 2018.")

# One annual edge records that a reviewer reviewed at least one recipe by a contributor.
reviewer_contributor_edges = network_review_events[
    ["year", "user_id", "contributor_id"]
].drop_duplicates()
edge_first_seen = (
    reviewer_contributor_edges.groupby(["user_id", "contributor_id"], as_index=False)["year"]
    .min()
    .rename(columns={"year": "first_seen_year"})
)
reviewer_first_seen = network_review_events.groupby("user_id")["year"].min()
contributor_first_seen = network_review_events.groupby("contributor_id")["year"].min()

network_growth_rows = []
cumulative_reviewers = 0
cumulative_contributors = 0
cumulative_ties = 0
for year in network_years:
    annual_events = network_review_events.loc[network_review_events["year"] == year]
    annual_edges = reviewer_contributor_edges.loc[reviewer_contributor_edges["year"] == year]
    new_reviewers = int(reviewer_first_seen.eq(year).sum())
    new_contributors = int(contributor_first_seen.eq(year).sum())
    new_ties = int(edge_first_seen["first_seen_year"].eq(year).sum())
    cumulative_reviewers += new_reviewers
    cumulative_contributors += new_contributors
    cumulative_ties += new_ties
    network_growth_rows.append(
        {
            "year": year,
            "review_events": len(annual_events),
            "active_reviewers": annual_events["user_id"].nunique(),
            "active_contributors": annual_events["contributor_id"].nunique(),
            "annual_reviewer_contributor_ties": len(annual_edges),
            "new_reviewers": new_reviewers,
            "new_contributors": new_contributors,
            "new_ties": new_ties,
            "cumulative_reviewers": cumulative_reviewers,
            "cumulative_contributors": cumulative_contributors,
            "cumulative_nodes_by_role": cumulative_reviewers + cumulative_contributors,
            "cumulative_ties": cumulative_ties,
        }
    )
network_growth = pd.DataFrame(network_growth_rows)

# A persistent reviewer-contributor pair contributes one authority connection once.
authority_increments = (
    edge_first_seen.groupby(["contributor_id", "first_seen_year"]).size().unstack(fill_value=0)
)
authority_increments = authority_increments.reindex(columns=network_years, fill_value=0)
authority_history = authority_increments.cumsum(axis=1)
authority_history.columns = network_years

top_authority_rows = []
first_top_ten_year = {}
for year in network_years:
    authority_counts = authority_history[year]
    ranked = authority_counts.loc[authority_counts > 0].rename("cumulative_unique_reviewers").reset_index()
    ranked["rank"] = ranked["cumulative_unique_reviewers"].rank(
        method="min", ascending=False
    ).astype(int)
    ranked = ranked.sort_values(
        ["rank", "contributor_id"], ascending=[True, True], kind="stable"
    ).head(10)
    for contributor_id in ranked["contributor_id"]:
        first_top_ten_year.setdefault(str(contributor_id), year)
    ranked.insert(0, "year", year)
    top_authority_rows.extend(ranked.to_dict("records"))
yearly_top_authorities = pd.DataFrame(top_authority_rows)

final_counts = authority_history[NETWORK_END_YEAR]
final_authorities = final_counts.loc[final_counts > 0].rename(
    "distinct_reviewers_through_2018"
).reset_index()
final_authorities = final_authorities.sort_values(
    ["distinct_reviewers_through_2018", "contributor_id"],
    ascending=[False, True],
    kind="stable",
).head(10).reset_index(drop=True)
final_authorities.insert(0, "rank_2018", range(1, len(final_authorities) + 1))
final_authorities["first_review_year"] = final_authorities["contributor_id"].map(
    contributor_first_seen
)
final_authorities["first_cumulative_top10_year"] = final_authorities[
    "contributor_id"
].map(first_top_ten_year)
review_totals = network_review_events.groupby("contributor_id").size()
final_authorities["review_events_through_2018"] = final_authorities["contributor_id"].map(
    review_totals
)

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
axes[0, 0].plot(network_growth["year"], network_growth["active_reviewers"], marker="o", label="Reviewers")
axes[0, 0].plot(network_growth["year"], network_growth["active_contributors"], marker="o", label="Contributors")
axes[0, 0].set_title("Active participants per year")
axes[0, 0].legend()
axes[0, 1].plot(network_growth["year"], network_growth["review_events"], marker="o", color="tab:green")
axes[0, 1].set_title("Distinct review events per year")
axes[1, 0].plot(network_growth["year"], network_growth["cumulative_nodes_by_role"], marker="o", color="tab:purple")
axes[1, 0].set_title("Cumulative reviewer + contributor nodes")
axes[1, 1].plot(network_growth["year"], network_growth["cumulative_ties"], marker="o", color="tab:red")
axes[1, 1].set_title("Cumulative reviewer-contributor ties")
for axis in axes.flat:
    axis.set_xlabel("Review year")
    axis.grid(True, alpha=0.3)
fig.suptitle("Food.com reviewer-contributor network growth, 2000-2018")
fig.tight_layout()
growth_figure_path = figures_dir / "reviewer_contributor_network_growth.png"
fig.savefig(growth_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

fig, axis = plt.subplots(figsize=(13, 7))
for contributor_id in final_authorities["contributor_id"]:
    axis.plot(
        network_years,
        authority_history.loc[contributor_id, network_years],
        marker="o",
        label=f"Contributor {contributor_id}",
    )
axis.set_title("Cumulative distinct reviewers for 2018 top authorities")
axis.set_xlabel("Review year")
axis.set_ylabel("Distinct reviewers accumulated")
axis.grid(True, alpha=0.3)
axis.legend(title="2018 authority rank", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
authority_figure_path = figures_dir / "reviewer_contributor_authority_emergence.png"
fig.savefig(authority_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

growth_csv_path = figures_dir / "reviewer_contributor_network_growth.csv"
top_authorities_csv_path = figures_dir / "reviewer_contributor_yearly_top_authorities.csv"
final_authorities_csv_path = figures_dir / "reviewer_contributor_top_authorities_2018.csv"
authority_history_csv_path = figures_dir / "reviewer_contributor_authority_history.csv"
network_growth.to_csv(growth_csv_path, index=False)
yearly_top_authorities.to_csv(top_authorities_csv_path, index=False)
final_authorities.to_csv(final_authorities_csv_path, index=False)
authority_history.rename_axis("contributor_id").reset_index().to_csv(
    authority_history_csv_path, index=False
)

print(f"Review events analyzed: {len(network_review_events):,}")
print(f"Review years: {NETWORK_START_YEAR}-{NETWORK_END_YEAR}")
print(f"Contributor attribution unmatched: {unmatched_recipe_count}")
print("2018 top authorities, ranked by cumulative distinct reviewers:")
display(final_authorities)
print("Top authorities by cumulative distinct reviewers in each year:")
display(yearly_top_authorities)
print(f"Saved network growth figure: {growth_figure_path}")
print(f"Saved authority emergence figure: {authority_figure_path}")
print(f"Saved network growth data: {growth_csv_path}")
print(f"Saved yearly top-authority data: {top_authorities_csv_path}")
print(f"Saved 2018 top-authority data: {final_authorities_csv_path}")
print(f"Saved cumulative authority history: {authority_history_csv_path}")


### Initial findings from the observed data

The network grows rapidly from its small 2000 baseline (108 review events, 82 active reviewers, 68 active contributors, and 107 reviewer-contributor ties). Annual review events and ties peak in 2008 at 167,295 events and 138,675 ties; active reviewers and contributors peak in 2009 at 36,284 and 12,920. Activity then falls overall, with a rebound in active reviewers in 2017, so the annual activity is not a monotonic growth curve. By 2018, the cumulative network contains 226,570 reviewer-role nodes, 27,926 contributor-role nodes, and 872,798 distinct reviewer-contributor ties.

Contributor `89831` ranks first in 2018 with 14,386 distinct reviewers. This contributor first appears in the review data in 2002 and first enters the cumulative top ten in 2005. Across the 2018 top ten, first top-ten entry years range from 2001 to 2011. The accompanying tables show the yearly leaders and each final leader's emergence year.

These are observed dataset counts, not a census of all Food.com activity. The decline in annual activity may reflect data coverage or platform activity, and should not by itself be interpreted as contributors losing influence. A cumulative top-ten entry year is a transparent ranking threshold, not a causal or statistical change point.

## Reviewer cohort retention and repeat contributor ties

Reviewer cohorts are defined by the year a user first appears reviewing in the dataset. The heatmap shows the percentage of each cohort that reviewed at least one recipe in each subsequent year, relative to the cohort's original size. Cohorts near 2018 are right-censored because later years are not available. For tie formation, a reviewer-contributor pair is new in its first observed review year; its recipe-review events in later years are repeat-tie events. Multiple recipes reviewed by the same reviewer from the same contributor in the pair's first year are counted as new-tie review events for this first-observed-tie measure. These are dataset-observed return and tie patterns, not necessarily users' complete Food.com histories.

In [ ]:
reviewer_year_activity = (
    network_review_events.groupby(["user_id", "year"])
    .size()
    .rename("review_events")
    .reset_index()
)
reviewer_first_year = reviewer_year_activity.groupby("user_id")["year"].min()
reviewer_cohort_sizes = reviewer_first_year.value_counts().sort_index()
cohort_years = list(range(NETWORK_START_YEAR, NETWORK_END_YEAR + 1))
reviewer_retention_matrix = pd.DataFrame(
    float("nan"),
    index=pd.Index(cohort_years, name="first_review_year"),
    columns=pd.Index(cohort_years, name="calendar_year"),
)
reviewer_cohort_rows = []
for cohort_year in cohort_years:
    cohort_reviewers = set(reviewer_first_year.index[reviewer_first_year == cohort_year])
    cohort_size = len(cohort_reviewers)
    if not cohort_size:
        continue
    for calendar_year in range(cohort_year, NETWORK_END_YEAR + 1):
        active_reviewers = set(
            reviewer_year_activity.loc[
                reviewer_year_activity["year"] == calendar_year, "user_id"
            ]
        )
        active_cohort_reviewers = len(cohort_reviewers & active_reviewers)
        retention_percent = 100 * active_cohort_reviewers / cohort_size
        reviewer_retention_matrix.loc[cohort_year, calendar_year] = retention_percent
        reviewer_cohort_rows.append(
            {
                "first_review_year": cohort_year,
                "calendar_year": calendar_year,
                "years_since_first_review": calendar_year - cohort_year,
                "cohort_size": cohort_size,
                "active_cohort_reviewers": active_cohort_reviewers,
                "retention_percent": retention_percent,
            }
        )
reviewer_cohort_retention = pd.DataFrame(reviewer_cohort_rows)

fig, axis = plt.subplots(figsize=(13, 9))
masked_retention = reviewer_retention_matrix.to_numpy(dtype=float)
image = axis.imshow(masked_retention, cmap="YlGnBu", vmin=0, vmax=100, aspect="auto")
axis.set_xticks(range(len(cohort_years)), cohort_years, rotation=45, ha="right")
axis.set_yticks(range(len(cohort_years)), cohort_years)
axis.set_xlabel("Calendar year of review")
axis.set_ylabel("Reviewer cohort (first review year)")
axis.set_title("Reviewer cohort retention: share active in each later year")
colorbar = fig.colorbar(image, ax=axis)
colorbar.set_label("Cohort reviewers active (%)")
fig.tight_layout()
reviewer_cohort_heatmap_path = figures_dir / "reviewer_cohort_retention_heatmap.png"
fig.savefig(reviewer_cohort_heatmap_path, dpi=200, bbox_inches="tight")
plt.close(fig)

annual_reviewer_sets = {
    int(year): set(group["user_id"])
    for year, group in reviewer_year_activity.groupby("year")
}
annual_reviewer_rows = []
previous_year_reviewers = set()
seen_reviewers = set()
for year in cohort_years:
    active_reviewers = annual_reviewer_sets.get(year, set())
    returning_reviewers = active_reviewers & previous_year_reviewers
    new_reviewers = active_reviewers - seen_reviewers
    annual_reviewer_rows.append(
        {
            "year": year,
            "active_reviewers": len(active_reviewers),
            "new_reviewers": len(new_reviewers),
            "returning_from_previous_year": len(returning_reviewers),
            "previous_year_active_reviewers": len(previous_year_reviewers),
            "year_over_year_reviewer_retention_pct": (
                100 * len(returning_reviewers) / len(previous_year_reviewers)
                if previous_year_reviewers
                else float("nan")
            ),
        }
    )
    previous_year_reviewers = active_reviewers
    seen_reviewers.update(active_reviewers)
annual_reviewer_retention = pd.DataFrame(annual_reviewer_rows)

tie_first_year = (
    network_review_events.groupby(["user_id", "contributor_id"])["year"]
    .min()
    .rename("tie_first_year")
    .reset_index()
)
tie_labeled_review_events = network_review_events.merge(
    tie_first_year,
    on=["user_id", "contributor_id"],
    how="left",
    validate="many_to_one",
)
tie_labeled_review_events["tie_status"] = "repeat_tie"
tie_labeled_review_events.loc[
    tie_labeled_review_events["year"] == tie_labeled_review_events["tie_first_year"],
    "tie_status",
] = "new_tie"
annual_tie_event_counts = (
    tie_labeled_review_events.groupby(["year", "tie_status"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=cohort_years, columns=["new_tie", "repeat_tie"], fill_value=0)
)
annual_tie_pair_counts = (
    tie_labeled_review_events[["year", "tie_status", "user_id", "contributor_id"]]
    .drop_duplicates()
    .groupby(["year", "tie_status"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=cohort_years, columns=["new_tie", "repeat_tie"], fill_value=0)
)
annual_tie_summary = pd.DataFrame(index=cohort_years)
annual_tie_summary.index.name = "year"
annual_tie_summary["review_events"] = annual_tie_event_counts.sum(axis=1)
annual_tie_summary["new_tie_review_events"] = annual_tie_event_counts["new_tie"]
annual_tie_summary["repeat_tie_review_events"] = annual_tie_event_counts["repeat_tie"]
annual_tie_summary["new_reviewer_contributor_ties"] = annual_tie_pair_counts["new_tie"]
annual_tie_summary["previously_seen_ties_active"] = annual_tie_pair_counts["repeat_tie"]
annual_tie_summary["new_tie_review_event_pct"] = (
    100 * annual_tie_summary["new_tie_review_events"] / annual_tie_summary["review_events"]
)
annual_tie_summary["repeat_tie_review_event_pct"] = (
    100 * annual_tie_summary["repeat_tie_review_events"] / annual_tie_summary["review_events"]
)
annual_tie_summary = annual_tie_summary.reset_index().merge(
    annual_reviewer_retention[["year", "active_reviewers", "new_reviewers", "returning_from_previous_year", "year_over_year_reviewer_retention_pct"]],
    on="year",
    how="left",
    validate="one_to_one",
)

fig, axis = plt.subplots(figsize=(12, 6))
axis.bar(
    annual_tie_summary["year"],
    annual_tie_summary["new_tie_review_events"],
    label="Review events on newly formed reviewer-contributor ties",
    color="tab:blue",
)
axis.bar(
    annual_tie_summary["year"],
    annual_tie_summary["repeat_tie_review_events"],
    bottom=annual_tie_summary["new_tie_review_events"],
    label="Review events on ties seen in an earlier year",
    color="tab:orange",
)
axis.set_title("Annual review events: new versus previously observed contributor ties")
axis.set_xlabel("Review year")
axis.set_ylabel("Review events")
axis.legend()
axis.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
tie_event_figure_path = figures_dir / "new_vs_repeat_contributor_tie_review_events.png"
fig.savefig(tie_event_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

reviewer_cohort_retention_path = figures_dir / "reviewer_cohort_retention.csv"
annual_reviewer_retention_path = figures_dir / "annual_reviewer_retention.csv"
annual_tie_summary_path = figures_dir / "new_vs_repeat_contributor_tie_review_events.csv"
reviewer_cohort_retention.to_csv(reviewer_cohort_retention_path, index=False)
annual_reviewer_retention.to_csv(annual_reviewer_retention_path, index=False)
annual_tie_summary.to_csv(annual_tie_summary_path, index=False)
print(f"Saved reviewer cohort heatmap: {reviewer_cohort_heatmap_path}")
print(f"Saved reviewer cohort retention data: {reviewer_cohort_retention_path}")
print(f"Saved annual reviewer retention data: {annual_reviewer_retention_path}")
print(f"Saved new versus repeat tie event figure: {tie_event_figure_path}")
print(f"Saved new versus repeat tie event data: {annual_tie_summary_path}")
display(annual_reviewer_retention)
display(annual_tie_summary.round(2))

### Interpretation notes

The heatmap denominator is each cohort's full first-review-year size, so a cell answers: what percentage of people first observed in year X reviewed again in year Y? The diagonal is 100% by definition. Blank cells before a cohort starts are not zero retention. The annual returning-reviewer percentage instead uses the previous year's active reviewers as its denominator and therefore describes year-to-year return among last year's active users.

A new reviewer-contributor tie means a reviewer is observed reviewing that contributor for the first time in the available 2000-2018 data; it does not mean the reviewer contributed a new recipe or that the pair never interacted before the dataset begins. Repeat-tie review events are events in later years on a pair already seen in an earlier year. A decline in review events can result from fewer active reviewers, fewer first-time ties, less activity on existing ties, or changes in data coverage; these tables help separate these patterns but cannot establish their cause.

### What the observed data shows

Year-over-year reviewer retention peaked at about 38.3% in 2006 and then fell to 33.4% in 2009, 27.4% in 2010, and 4.4% in 2018. For the 2008 first-review cohort, 21.6% reviewed again in 2009, 12.9% in 2010, and 0.4% in 2018. This indicates low observed return, especially in the later years, though users missing from the dataset may still have been active outside its coverage. Of 17,339 reviewers active in 2018, 15,570 were first observed in 2018 and 968 had also reviewed in 2017.

Review activity peaked at 159,445 events in 2009 and fell to 20,647 in 2018 (about 87% lower). In 2009, 15.3% of review events were on reviewer-contributor ties observed in an earlier year; in 2018 that share was 4.6%, with the remaining 95.4% on ties first observed in 2018. This suggests that the observed later-period activity relied increasingly on new-to-the-dataset reviewer-contributor pairs rather than repeat events on older ties. It helps describe the post-2009 decline, but cannot distinguish platform-wide churn from incomplete dataset coverage or prove why reviewers left.

## Year-to-year stability of top authorities

For each review year, rank contributors by the number of distinct reviewers who reviewed at least one of their recipes that year. Compare adjacent annual top-10 lists using Jaccard similarity (shared contributors divided by contributors in either list). Also calculate Spearman rank correlation among contributors appearing in both top-10 lists; this rank statistic is descriptive and is shown alongside the number of shared contributors. It is omitted when fewer than three contributors are shared because correlations with only one or two observations are not informative. Ties in reviewer counts are resolved deterministically by contributor ID.

In [ ]:
TOP_AUTHORITIES_PER_YEAR = 10
annual_authority_counts = (
    network_review_events.groupby(["year", "contributor_id"])["user_id"]
    .nunique()
    .rename("distinct_reviewers")
    .reset_index()
)
annual_authority_counts["contributor_id"] = annual_authority_counts["contributor_id"].astype(str)
yearly_authority_rankings = (
    annual_authority_counts.sort_values(
        ["year", "distinct_reviewers", "contributor_id"],
        ascending=[True, False, True],
        kind="stable",
    )
    .groupby("year", sort=True, group_keys=False)
    .head(TOP_AUTHORITIES_PER_YEAR)
    .copy()
    .reset_index(drop=True)
)
yearly_authority_rankings["rank"] = yearly_authority_rankings.groupby("year").cumcount() + 1

yearly_authority_sets = {
    int(year): set(group["contributor_id"])
    for year, group in yearly_authority_rankings.groupby("year")
}
yearly_rank_maps = {
    int(year): group.set_index("contributor_id")["rank"].to_dict()
    for year, group in yearly_authority_rankings.groupby("year")
}
authority_overlap_rows = []
for previous_year in range(NETWORK_START_YEAR, NETWORK_END_YEAR):
    current_year = previous_year + 1
    previous_top = yearly_authority_sets.get(previous_year, set())
    current_top = yearly_authority_sets.get(current_year, set())
    common_contributors = previous_top & current_top
    union_contributors = previous_top | current_top
    shared_count = len(common_contributors)
    if shared_count >= 3:
        previous_ranks = pd.Series(
            {contributor: yearly_rank_maps[previous_year][contributor] for contributor in common_contributors}
        )
        current_ranks = pd.Series(
            {contributor: yearly_rank_maps[current_year][contributor] for contributor in common_contributors}
        )
        rank_correlation = float(previous_ranks.corr(current_ranks, method="pearson"))
    else:
        rank_correlation = float("nan")
    authority_overlap_rows.append(
        {
            "previous_year": previous_year,
            "year": current_year,
            "year_transition": f"{previous_year}-{current_year}",
            "shared_top_authorities": shared_count,
            "jaccard_similarity": shared_count / len(union_contributors) if union_contributors else float("nan"),
            "spearman_rank_correlation_shared": rank_correlation,
        }
    )
authority_overlap = pd.DataFrame(authority_overlap_rows)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(
    authority_overlap["year_transition"],
    authority_overlap["jaccard_similarity"],
    marker="o",
    color="tab:blue",
)
axes[0].set_title("Top-10 authority membership overlap (Jaccard similarity)")
axes[0].set_ylabel("Jaccard similarity")
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)
axes[1].plot(
    authority_overlap["year_transition"],
    authority_overlap["spearman_rank_correlation_shared"],
    marker="o",
    color="tab:orange",
)
axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.6)
axes[1].set_title("Rank stability among contributors shared by adjacent top tens")
axes[1].set_ylabel("Spearman rank correlation")
axes[1].set_xlabel("Adjacent review-year transition")
axes[1].set_ylim(-1.05, 1.05)
axes[1].grid(True, alpha=0.3)
for axis in axes:
    axis.tick_params(axis="x", labelrotation=60)
fig.suptitle("Year-to-year stability of Food.com top authorities")
fig.tight_layout()
authority_overlap_figure_path = figures_dir / "yearly_top_authority_overlap.png"
fig.savefig(authority_overlap_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

yearly_authority_rankings_path = figures_dir / "yearly_top_authority_rankings.csv"
authority_overlap_path = figures_dir / "yearly_top_authority_overlap.csv"
yearly_authority_rankings.to_csv(yearly_authority_rankings_path, index=False)
authority_overlap.to_csv(authority_overlap_path, index=False)
display(yearly_authority_rankings)
display(authority_overlap.round(3))
print(f"Saved yearly top-authority rankings: {yearly_authority_rankings_path}")
print(f"Saved year-to-year overlap metrics: {authority_overlap_path}")
print(f"Saved overlap chart: {authority_overlap_figure_path}")

### Interpretation notes

Jaccard similarity measures continuity in who makes the annual top ten, while Spearman correlation measures whether those shared contributors keep similar positions. The two measures can differ: two years may share most names but still reorder them. Jaccard should be interpreted alongside the shared-member count; rank correlation is omitted when fewer than three contributors overlap. Rankings are based on annual distinct reviewers, so they differ from the earlier cumulative authority ranking. They represent reviewer attention in the observed review data, not a complete measure of influence.

Across the 18 year-to-year transitions, the mean Jaccard similarity is about 0.67, with an average of 7.8 contributors shared between adjacent top tens. The top ten is therefore fairly persistent after the earliest years. Membership overlap alone does not capture rank changes: for example, 2002-2003 shares 9 of 10 authorities (Jaccard 0.82), but their Spearman rank correlation is only about 0.13, indicating that the shared contributors shifted positions substantially. The 2000-2001 comparison has only two shared authorities, so rank correlation is omitted.

## Inequality in review distribution across contributors

For every review year, calculate the Gini coefficient of review counts across contributors who received at least one review, plus the share of all reviews received by the top 1% and top 10% of those active contributors. Review counts are assigned to a recipe's contributor and count reviewer-recipe events; repeated reviews by the same reviewer of different recipes count separately. The top share uses the ceiling of the percentage of active contributors, with at least one contributor. This describes concentration of observed review activity. Increasing concentration can be consistent with a rich-get-richer pattern, but does not by itself establish preferential attachment or causality.

In [ ]:
import math

TOP_REVIEWER_SHARE_FRACTIONS = (0.01, 0.10)
annual_contributor_reviews = (
    network_review_events.groupby(["year", "contributor_id"])
    .size()
    .rename("review_events")
    .reset_index()
)
concentration_rows = []
for year in network_years:
    counts = annual_contributor_reviews.loc[
        annual_contributor_reviews["year"] == year, "review_events"
    ].to_numpy(dtype=float)
    counts.sort()
    active_contributors = len(counts)
    total_reviews = float(counts.sum())
    if not active_contributors or total_reviews <= 0:
        raise ValueError(f"No contributor review activity was found for {year}.")
    gini = (
        2 * float(sum((index + 1) * count for index, count in enumerate(counts)))
        / (active_contributors * total_reviews)
        - (active_contributors + 1) / active_contributors
    )
    row = {
        "year": year,
        "active_contributors": active_contributors,
        "total_review_events": int(total_reviews),
        "gini_review_count": gini,
    }
    for fraction in TOP_REVIEWER_SHARE_FRACTIONS:
        contributor_count = max(1, math.ceil(fraction * active_contributors))
        share_percent = 100 * float(counts[-contributor_count:].sum()) / total_reviews
        percentage_label = int(fraction * 100)
        row[f"top_{percentage_label}_percent_contributors"] = contributor_count
        row[f"top_{percentage_label}_percent_review_share"] = share_percent
    concentration_rows.append(row)

review_concentration = pd.DataFrame(concentration_rows)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(
    review_concentration["year"],
    review_concentration["gini_review_count"],
    marker="o",
    color="tab:purple",
)
axes[0].set_title("Inequality in annual reviews received by contributors")
axes[0].set_ylabel("Gini coefficient")
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)
axes[1].plot(
    review_concentration["year"],
    review_concentration["top_1_percent_review_share"],
    marker="o",
    label="Top 1% of active contributors",
)
axes[1].plot(
    review_concentration["year"],
    review_concentration["top_10_percent_review_share"],
    marker="o",
    label="Top 10% of active contributors",
)
axes[1].set_title("Share of annual review events going to top contributors")
axes[1].set_xlabel("Review year")
axes[1].set_ylabel("Share of review events (%)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
low_volume_years = review_concentration.loc[
    review_concentration["total_review_events"] < 25000, "year"
]
if not low_volume_years.empty:
    low_volume_start_year = int(low_volume_years.min())
    for axis in axes:
        axis.axvline(
            low_volume_start_year,
            color="black",
            linestyle="--",
            alpha=0.6,
        )
    axes[1].axvspan(
        low_volume_start_year,
        max(review_concentration["year"]),
        color="gray",
        alpha=0.08,
        label="Years with <25,000 reviews",
    )
    axes[1].legend()
fig.suptitle("Annual review concentration among Food.com contributors")
fig.tight_layout()
review_concentration_figure_path = figures_dir / "annual_contributor_review_concentration.png"
fig.savefig(review_concentration_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

review_concentration_path = figures_dir / "annual_contributor_review_concentration.csv"
review_concentration.to_csv(review_concentration_path, index=False)
display(review_concentration.round(3))
peak_gini_row = review_concentration.loc[review_concentration["gini_review_count"].idxmax()]
print(
    f"Peak annual inequality: Gini={peak_gini_row['gini_review_count']:.3f} "
    f"in {int(peak_gini_row['year'])}."
)
print(f"Saved concentration chart: {review_concentration_figure_path}")
print(f"Saved concentration data: {review_concentration_path}")

### Interpretation notes

A higher Gini means annual reviews are distributed less evenly across contributors; a higher top-1% or top-10% share means that group received a larger share of reviews. The measure includes only contributors who received at least one review that year, so it describes inequality among active recipients, not among every person who ever contributed a recipe. The earliest years have very few active contributors, and later low-volume years have fewer review events, so the endpoints should be interpreted cautiously.

In the observed data, concentration rose as the community expanded during the early and middle years: the Gini increased from 0.311 in 2000 to a peak of 0.825 in 2008, when the top 10% of active contributors received about 78.0% of reviews. It then generally declined; by 2018 the Gini was 0.652 and the top 10% share was 57.9%. This is evidence that review distribution became more unequal during the growth phase, but it did not continue becoming more unequal through 2018.

These metrics show concentration, not why it occurs. To test preferential attachment directly, a follow-up analysis should model whether a contributor's prior reviewer degree predicts new reviewers in the next period while accounting for exposure (such as the number of available recipes), contributor age, and activity. A rising Gini alone is not evidence that popularity caused further popularity.

## Egocentric networks of prominent contributors

The three highest-ranked contributors in the cumulative 2018 authority table are shown as egos. Following the reference diagram, each two-hop network links the contributor (ego) to reviewers (first-order alters), then to other contributors whose recipes those reviewers also reviewed (second-order alters). To keep the plots legible, each ego is limited to the 8 reviewers with the most distinct recipes reviewed from that ego, and up to 3 other contributors per selected reviewer, ranked by distinct recipes. The display is a bounded neighborhood, not the ego's complete network. Edges represent review activity from 2000 through 2018; the two-hop view treats connections as undirected for visualization.

In [ ]:
import math

EGO_CONTRIBUTOR_COUNT = 3
MAX_REVIEWER_ALTERS = 8
MAX_CONTRIBUTOR_ALTERS_PER_REVIEWER = 3

if final_authorities.empty:
    raise ValueError("No prominent contributors are available to build ego networks.")

pair_recipe_counts = (
    network_review_events.groupby(["user_id", "contributor_id"])["recipe_id"]
    .nunique()
    .rename("distinct_recipes")
    .reset_index()
)
prominent_ego_ids = final_authorities["contributor_id"].head(EGO_CONTRIBUTOR_COUNT).astype(str).tolist()
egonet_edge_rows = []
egonet_node_rows = []

for ego_id in prominent_ego_ids:
    ego_reviewers = pair_recipe_counts.loc[
        pair_recipe_counts["contributor_id"].astype(str) == ego_id
    ].sort_values(
        ["distinct_recipes", "user_id"], ascending=[False, True], kind="stable"
    ).head(MAX_REVIEWER_ALTERS)
    if ego_reviewers.empty:
        raise ValueError(f"Contributor {ego_id} has no reviewer alters in the selected period.")

    selected_reviewers = set(ego_reviewers["user_id"].astype(str))
    ego_contributor_alters = {}
    for reviewer_id in sorted(selected_reviewers):
        reviewer_contributors = pair_recipe_counts.loc[
            (pair_recipe_counts["user_id"].astype(str) == reviewer_id)
            & (pair_recipe_counts["contributor_id"].astype(str) != ego_id)
        ].sort_values(
            ["distinct_recipes", "contributor_id"],
            ascending=[False, True],
            kind="stable",
        ).head(MAX_CONTRIBUTOR_ALTERS_PER_REVIEWER)

        for row in reviewer_contributors.itertuples(index=False):
            alter_id = str(row.contributor_id)
            ego_contributor_alters[alter_id] = "contributor"
            egonet_edge_rows.append(
                {
                    "ego_contributor_id": ego_id,
                    "source_id": f"reviewer:{reviewer_id}",
                    "target_id": f"contributor:{alter_id}",
                    "source_type": "reviewer",
                    "target_type": "contributor_alter",
                    "degree_level": 2,
                    "distinct_recipes": int(row.distinct_recipes),
                }
            )

    egonet_nodes = [
        {"ego_contributor_id": ego_id, "node_id": ego_id, "node_type": "ego_contributor", "degree_level": 0}
    ]
    egonet_nodes.extend(
        {
            "ego_contributor_id": ego_id,
            "node_id": str(row.user_id),
            "node_type": "reviewer_alter",
            "degree_level": 1,
        }
        for row in ego_reviewers.itertuples(index=False)
    )
    egonet_nodes.extend(
        {
            "ego_contributor_id": ego_id,
            "node_id": alter_id,
            "node_type": "contributor_alter",
            "degree_level": 2,
        }
        for alter_id in sorted(ego_contributor_alters)
    )
    egonet_node_rows.extend(egonet_nodes)

    egonet_edge_rows.extend(
        {
            "ego_contributor_id": ego_id,
            "source_id": f"contributor:{ego_id}",
            "target_id": f"reviewer:{row.user_id}",
            "source_type": "ego_contributor",
            "target_type": "reviewer_alter",
            "degree_level": 1,
            "distinct_recipes": int(row.distinct_recipes),
        }
        for row in ego_reviewers.itertuples(index=False)
    )

    contributor_ids = sorted(ego_contributor_alters)
    ego_node_key = f"contributor:{ego_id}"
    reviewer_node_keys = {reviewer_id: f"reviewer:{reviewer_id}" for reviewer_id in selected_reviewers}
    contributor_node_keys = {
        contributor_id: f"contributor:{contributor_id}" for contributor_id in contributor_ids
    }
    reviewer_angles = {
        str(reviewer_id): (2 * math.pi * index / len(selected_reviewers))
        for index, reviewer_id in enumerate(sorted(selected_reviewers))
    }
    contributor_angles = {
        contributor_id: (2 * math.pi * index / max(1, len(contributor_ids)))
        for index, contributor_id in enumerate(contributor_ids)
    }
    positions = {ego_node_key: (0.0, 0.0)}
    positions.update(
        {
            reviewer_node_keys[reviewer_id]: (math.cos(angle), math.sin(angle))
            for reviewer_id, angle in reviewer_angles.items()
        }
    )
    positions.update(
        {
            contributor_node_keys[contributor_id]: (2.1 * math.cos(angle), 2.1 * math.sin(angle))
            for contributor_id, angle in contributor_angles.items()
        }
    )

    fig, axis = plt.subplots(figsize=(11, 9))
    current_edges = [row for row in egonet_edge_rows if row["ego_contributor_id"] == ego_id]
    max_weight = max(row["distinct_recipes"] for row in current_edges)
    for edge in current_edges:
        x_start, y_start = positions[edge["source_id"]]
        x_end, y_end = positions[edge["target_id"]]
        linewidth = 0.7 + 2.2 * edge["distinct_recipes"] / max_weight
        color = "#667085" if edge["degree_level"] == 1 else "#98A2B3"
        axis.plot([x_start, x_end], [y_start, y_end], color=color, linewidth=linewidth, alpha=0.65, zorder=1)

    axis.scatter([0], [0], s=900, marker="*", color="#D92D20", edgecolor="black", zorder=3, label="Ego contributor")
    reviewer_ids = sorted(selected_reviewers)
    axis.scatter(
        [positions[reviewer_node_keys[node_id]][0] for node_id in reviewer_ids],
        [positions[reviewer_node_keys[node_id]][1] for node_id in reviewer_ids],
        s=430,
        color="#84CAFF",
        edgecolor="#175CD3",
        zorder=3,
        label="Reviewer alters",
    )
    if contributor_ids:
        axis.scatter(
            [positions[contributor_node_keys[node_id]][0] for node_id in contributor_ids],
            [positions[contributor_node_keys[node_id]][1] for node_id in contributor_ids],
            s=360,
            marker="s",
            color="#A6F4C5",
            edgecolor="#039855",
            zorder=3,
            label="Other contributor alters",
        )
    axis.annotate(f"C {ego_id}", (0.0, 0.0), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=8)
    for reviewer_id in reviewer_ids:
        x_pos, y_pos = positions[reviewer_node_keys[reviewer_id]]
        axis.annotate(f"R {reviewer_id}", (x_pos, y_pos), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=8)
    for contributor_id in contributor_ids:
        x_pos, y_pos = positions[contributor_node_keys[contributor_id]]
        axis.annotate(f"C {contributor_id}", (x_pos, y_pos), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=8)

    axis.set_title(f"Two-hop ego network: contributor {ego_id} (2000-2018)")
    axis.legend(loc="upper right")
    axis.set_aspect("equal")
    axis.axis("off")
    fig.tight_layout()
    ego_figure_path = figures_dir / f"contributor_{ego_id}_egocentric_network.png"
    fig.savefig(ego_figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved egocentric network for contributor {ego_id}: {ego_figure_path}")

egonet_nodes = pd.DataFrame(egonet_node_rows)
egonet_edges = pd.DataFrame(egonet_edge_rows)
egonet_nodes_path = figures_dir / "prominent_contributor_egonet_nodes.csv"
egonet_edges_path = figures_dir / "prominent_contributor_egonet_edges.csv"
egonet_nodes.to_csv(egonet_nodes_path, index=False)
egonet_edges.to_csv(egonet_edges_path, index=False)
print(f"Saved egonet nodes: {egonet_nodes_path}")
print(f"Saved egonet edges: {egonet_edges_path}")
display(egonet_nodes.groupby(["ego_contributor_id", "node_type"]).size().rename("nodes").reset_index())

## Ego network of the leading contributor across time slices

This section follows the 2018 top-ranked contributor across four review-year periods: 2000-2005, 2006-2010, 2011-2014, and 2015-2018. Each panel shows the contributor, up to 6 reviewers with the most distinct recipes from that contributor in the period, and up to 2 other contributors per selected reviewer. Edge widths reflect distinct recipes reviewed in that period. The same selection rule is applied in every panel; this is a readable sample of each period's neighborhood, not a complete network. Overall review activity is much lower in the final slice, so its smaller or changing network should be interpreted cautiously.

In [ ]:
import math

FOCAL_CONTRIBUTOR_ID = str(final_authorities.iloc[0]["contributor_id"])
TIME_SLICES = [
    ("2000-2005", 2000, 2005),
    ("2006-2010", 2006, 2010),
    ("2011-2014", 2011, 2014),
    ("2015-2018", 2015, 2018),
]
MAX_REVIEWERS_PER_SLICE = 6
MAX_OTHER_CONTRIBUTORS_PER_REVIEWER = 2
if FOCAL_CONTRIBUTOR_ID not in set(network_review_events["contributor_id"].astype(str)):
    raise ValueError(f"Focal contributor {FOCAL_CONTRIBUTOR_ID} has no matched review events.")

temporal_egonet_nodes = []
temporal_egonet_edges = []
temporal_slice_summary = []
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

for axis, (slice_label, start_year, end_year) in zip(axes.flat, TIME_SLICES):
    slice_events = network_review_events.loc[
        network_review_events["year"].between(start_year, end_year)
    ]
    if slice_events.empty:
        raise ValueError(f"No review events found in time slice {slice_label}.")
    slice_pair_counts = (
        slice_events.groupby(["user_id", "contributor_id"])["recipe_id"]
        .nunique()
        .rename("distinct_recipes")
        .reset_index()
    )
    ego_reviewers = slice_pair_counts.loc[
        slice_pair_counts["contributor_id"].astype(str) == FOCAL_CONTRIBUTOR_ID
    ].sort_values(
        ["distinct_recipes", "user_id"], ascending=[False, True], kind="stable"
    ).head(MAX_REVIEWERS_PER_SLICE)
    if ego_reviewers.empty:
        raise ValueError(
            f"Focal contributor {FOCAL_CONTRIBUTOR_ID} has no reviewers in {slice_label}."
        )

    reviewer_ids = ego_reviewers["user_id"].astype(str).tolist()
    reviewer_node_keys = {reviewer_id: f"reviewer:{reviewer_id}" for reviewer_id in reviewer_ids}
    reviewer_labels = {
        reviewer_id: f"R{index}"
        for index, reviewer_id in enumerate(reviewer_ids, start=1)
    }
    other_contributor_ids = set()
    slice_edges = []
    for reviewer_id in reviewer_ids:
        reviewer_contributors = slice_pair_counts.loc[
            (slice_pair_counts["user_id"].astype(str) == reviewer_id)
            & (slice_pair_counts["contributor_id"].astype(str) != FOCAL_CONTRIBUTOR_ID)
        ].sort_values(
            ["distinct_recipes", "contributor_id"],
            ascending=[False, True],
            kind="stable",
        ).head(MAX_OTHER_CONTRIBUTORS_PER_REVIEWER)
        for row in reviewer_contributors.itertuples(index=False):
            other_id = str(row.contributor_id)
            other_contributor_ids.add(other_id)
            slice_edges.append(
                {
                    "time_slice": slice_label,
                    "start_year": start_year,
                    "end_year": end_year,
                    "ego_contributor_id": FOCAL_CONTRIBUTOR_ID,
                    "source_id": f"reviewer:{reviewer_id}",
                    "target_id": f"contributor:{other_id}",
                    "source_type": "reviewer_alter",
                    "target_type": "contributor_alter",
                    "degree_level": 2,
                    "distinct_recipes": int(row.distinct_recipes),
                }
            )

    ego_node_key = f"contributor:{FOCAL_CONTRIBUTOR_ID}"
    contributor_node_keys = {
        contributor_id: f"contributor:{contributor_id}"
        for contributor_id in sorted(other_contributor_ids)
    }
    contributor_labels = {
        contributor_id: f"A{index}"
        for index, contributor_id in enumerate(sorted(other_contributor_ids), start=1)
    }
    for row in ego_reviewers.itertuples(index=False):
        slice_edges.append(
            {
                "time_slice": slice_label,
                "start_year": start_year,
                "end_year": end_year,
                "ego_contributor_id": FOCAL_CONTRIBUTOR_ID,
                "source_id": ego_node_key,
                "target_id": reviewer_node_keys[str(row.user_id)],
                "source_type": "ego_contributor",
                "target_type": "reviewer_alter",
                "degree_level": 1,
                "distinct_recipes": int(row.distinct_recipes),
            }
        )
    temporal_egonet_edges.extend(slice_edges)

    slice_nodes = [
        {
            "time_slice": slice_label,
            "ego_contributor_id": FOCAL_CONTRIBUTOR_ID,
            "node_id": ego_node_key,
            "label": "Ego",
            "node_type": "ego_contributor",
            "degree_level": 0,
        }
    ]
    slice_nodes.extend(
        {
            "time_slice": slice_label,
            "ego_contributor_id": FOCAL_CONTRIBUTOR_ID,
            "node_id": reviewer_node_keys[reviewer_id],
            "label": reviewer_labels[reviewer_id],
            "node_type": "reviewer_alter",
            "degree_level": 1,
        }
        for reviewer_id in reviewer_ids
    )
    slice_nodes.extend(
        {
            "time_slice": slice_label,
            "ego_contributor_id": FOCAL_CONTRIBUTOR_ID,
            "node_id": contributor_node_keys[contributor_id],
            "label": contributor_labels[contributor_id],
            "node_type": "contributor_alter",
            "degree_level": 2,
        }
        for contributor_id in sorted(other_contributor_ids)
    )
    temporal_egonet_nodes.extend(slice_nodes)

    reviewer_angles = {
        reviewer_id: 2 * math.pi * index / len(reviewer_ids)
        for index, reviewer_id in enumerate(reviewer_ids)
    }
    contributor_ids = sorted(other_contributor_ids)
    contributor_angles = {
        contributor_id: 2 * math.pi * index / max(1, len(contributor_ids))
        for index, contributor_id in enumerate(contributor_ids)
    }
    positions = {ego_node_key: (0.0, 0.0)}
    positions.update(
        {
            reviewer_node_keys[reviewer_id]: (1.0 * math.cos(angle), 1.0 * math.sin(angle))
            for reviewer_id, angle in reviewer_angles.items()
        }
    )
    positions.update(
        {
            contributor_node_keys[contributor_id]: (2.7 * math.cos(angle), 2.7 * math.sin(angle))
            for contributor_id, angle in contributor_angles.items()
        }
    )
    max_weight = max(edge["distinct_recipes"] for edge in slice_edges)
    for edge in slice_edges:
        start_x, start_y = positions[edge["source_id"]]
        end_x, end_y = positions[edge["target_id"]]
        color = "#667085" if edge["degree_level"] == 1 else "#98A2B3"
        axis.plot(
            [start_x, end_x], [start_y, end_y], color=color,
            linewidth=0.7 + 2.2 * edge["distinct_recipes"] / max_weight,
            alpha=0.65, zorder=1,
        )
    axis.scatter([0], [0], s=720, marker="*", color="#D92D20", edgecolor="black", zorder=3, label="Ego contributor")
    axis.scatter(
        [positions[reviewer_node_keys[node]][0] for node in reviewer_ids],
        [positions[reviewer_node_keys[node]][1] for node in reviewer_ids],
        s=350, color="#84CAFF", edgecolor="#175CD3", zorder=3, label="Reviewer alters",
    )
    if contributor_ids:
        axis.scatter(
            [positions[contributor_node_keys[node]][0] for node in contributor_ids],
            [positions[contributor_node_keys[node]][1] for node in contributor_ids],
            s=290, marker="s", color="#A6F4C5", edgecolor="#039855", zorder=3,
            label="Other contributor alters",
        )
    axis.annotate("Ego", (0, 0), xytext=(0, 11), textcoords="offset points", ha="center", fontsize=8, fontweight="bold")
    for reviewer_id in reviewer_ids:
        point = positions[reviewer_node_keys[reviewer_id]]
        axis.annotate(reviewer_labels[reviewer_id], point, xytext=(0, 10), textcoords="offset points", ha="center", fontsize=8)
    for contributor_id in contributor_ids:
        point = positions[contributor_node_keys[contributor_id]]
        axis.annotate(contributor_labels[contributor_id], point, xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8)
    slice_review_count = len(slice_events)
    slice_ego_recipes = int(
        slice_events.loc[
            slice_events["contributor_id"].astype(str) == FOCAL_CONTRIBUTOR_ID,
            "recipe_id",
        ].nunique()
    )
    axis.set_title(
        f"{slice_label}: {slice_ego_recipes} ego recipes, "
        f"{len(reviewer_ids)} reviewers shown\n{slice_review_count:,} review events on site"
    )
    axis.set_aspect("equal")
    axis.axis("off")
    temporal_slice_summary.append(
        {
            "time_slice": slice_label,
            "start_year": start_year,
            "end_year": end_year,
            "site_review_events": slice_review_count,
            "ego_recipes_reviewed": slice_ego_recipes,
            "ego_unique_reviewers": int(
                slice_events.loc[
                    slice_events["contributor_id"].astype(str) == FOCAL_CONTRIBUTOR_ID,
                    "user_id",
                ].nunique()
            ),
            "reviewers_shown": len(reviewer_ids),
            "other_contributors_shown": len(contributor_ids),
        }
    )

legend_handles, legend_labels = axes.flat[0].get_legend_handles_labels()
fig.legend(legend_handles, legend_labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 0.96), fontsize=9)
fig.suptitle(f"Contributor {FOCAL_CONTRIBUTOR_ID}: ego network across review-year slices", y=1.0)
fig.tight_layout(rect=(0, 0, 1, 0.91))
temporal_egonet_figure_path = figures_dir / f"contributor_{FOCAL_CONTRIBUTOR_ID}_egonet_over_time.png"
fig.savefig(temporal_egonet_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

temporal_egonet_node_data = pd.DataFrame(temporal_egonet_nodes)
temporal_egonet_edge_data = pd.DataFrame(temporal_egonet_edges)
temporal_egonet_summary = pd.DataFrame(temporal_slice_summary)
temporal_egonet_nodes_path = figures_dir / "top_contributor_egonet_over_time_nodes.csv"
temporal_egonet_edges_path = figures_dir / "top_contributor_egonet_over_time_edges.csv"
temporal_egonet_summary_path = figures_dir / "top_contributor_egonet_over_time_summary.csv"
temporal_egonet_node_data.to_csv(temporal_egonet_nodes_path, index=False)
temporal_egonet_edge_data.to_csv(temporal_egonet_edges_path, index=False)
temporal_egonet_summary.to_csv(temporal_egonet_summary_path, index=False)
display(temporal_egonet_summary)
print(f"Saved temporal ego-network figure: {temporal_egonet_figure_path}")
print(f"Saved temporal ego-network nodes: {temporal_egonet_nodes_path}")
print(f"Saved temporal ego-network edges: {temporal_egonet_edges_path}")

## Egocentric networks of prominent ingredients

Ingredient prominence is measured by the number of distinct recipes containing the ingredient. The three most frequent ingredients are used as egos. Each ego is connected to its ten most frequent co-ingredients; edge weight is the number of distinct recipes containing both ingredients. Ingredient strings are normalized to lowercase with repeated whitespace collapsed, but synonyms and spelling variants are not merged. These are one-hop ego networks, not ingredient nutrition or dietary classifications.

In [ ]:
import ast
import math
from collections import Counter
from itertools import combinations

INGREDIENT_EGO_COUNT = 3
MAX_INGREDIENT_ALTERS = 10
figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
ingredient_recipes_path = Path(dataset_path) / "RAW_recipes.csv"
if not ingredient_recipes_path.is_file():
    raise FileNotFoundError(f"RAW_recipes.csv was not found at {ingredient_recipes_path}")

ingredient_data = pd.read_csv(
    ingredient_recipes_path,
    usecols=["id", "ingredients"],
    dtype={"id": "string", "ingredients": "string"},
)
if ingredient_data["id"].isna().any() or ingredient_data["id"].duplicated().any():
    raise ValueError("Recipe IDs must be present and unique for ingredient network construction.")
if ingredient_data["ingredients"].isna().any():
    raise ValueError("Some recipes have missing ingredient lists; ingredient counts would be incomplete.")

ingredient_frequency = Counter()
ingredient_pair_frequency = Counter()
ingredient_parse_errors = []
for recipe_id, raw_ingredients in ingredient_data[["id", "ingredients"]].itertuples(index=False, name=None):
    try:
        parsed_ingredients = ast.literal_eval(raw_ingredients)
    except (SyntaxError, ValueError) as error:
        ingredient_parse_errors.append((recipe_id, str(error)))
        continue
    if not isinstance(parsed_ingredients, list) or not all(
        isinstance(ingredient, str) for ingredient in parsed_ingredients
    ):
        ingredient_parse_errors.append((recipe_id, "ingredient value is not a list of strings"))
        continue
    recipe_ingredients = sorted(
        {" ".join(ingredient.lower().split()) for ingredient in parsed_ingredients if ingredient.strip()}
    )
    ingredient_frequency.update(recipe_ingredients)
    ingredient_pair_frequency.update(combinations(recipe_ingredients, 2))

if ingredient_parse_errors:
    raise ValueError(
        f"Could not parse ingredient lists for {len(ingredient_parse_errors)} recipes; "
        f"first examples: {ingredient_parse_errors[:5]}"
    )
if not ingredient_frequency:
    raise ValueError("No ingredients were parsed from the recipe dataset.")

prominent_ingredients = sorted(
    ingredient_frequency.items(), key=lambda item: (-item[1], item[0])
)[:INGREDIENT_EGO_COUNT]
ingredient_node_rows = []
ingredient_edge_rows = []
fig, axes = plt.subplots(1, len(prominent_ingredients), figsize=(18, 7))
if len(prominent_ingredients) == 1:
    axes = [axes]

for axis, (ego_ingredient, ego_recipe_count) in zip(axes, prominent_ingredients):
    coingredients = [
        (ingredient_b if ingredient_a == ego_ingredient else ingredient_a, count)
        for (ingredient_a, ingredient_b), count in ingredient_pair_frequency.items()
        if ego_ingredient in (ingredient_a, ingredient_b)
    ]
    top_alters = sorted(coingredients, key=lambda item: (-item[1], item[0]))[:MAX_INGREDIENT_ALTERS]
    if not top_alters:
        raise ValueError(f"Ingredient {ego_ingredient!r} has no co-ingredient ties.")

    ingredient_node_rows.append(
        {
            "ego_ingredient": ego_ingredient,
            "ingredient": ego_ingredient,
            "node_type": "ego_ingredient",
            "recipe_count": ego_recipe_count,
        }
    )
    ingredient_node_rows.extend(
        {
            "ego_ingredient": ego_ingredient,
            "ingredient": alter,
            "node_type": "coingredient_alter",
            "recipe_count": ingredient_frequency[alter],
        }
        for alter, _ in top_alters
    )
    ingredient_edge_rows.extend(
        {
            "ego_ingredient": ego_ingredient,
            "alter_ingredient": alter,
            "recipes_containing_both": shared_recipe_count,
        }
        for alter, shared_recipe_count in top_alters
    )

    positions = {ego_ingredient: (0.0, 0.0)}
    for index, (alter, _) in enumerate(top_alters):
        angle = 2 * math.pi * index / len(top_alters)
        positions[alter] = (math.cos(angle), math.sin(angle))
    max_shared_recipes = max(count for _, count in top_alters)
    for alter, shared_recipe_count in top_alters:
        x_end, y_end = positions[alter]
        axis.plot(
            [0, x_end],
            [0, y_end],
            color="#98A2B3",
            linewidth=0.8 + 3.2 * shared_recipe_count / max_shared_recipes,
            alpha=0.7,
            zorder=1,
        )
    axis.scatter([0], [0], s=900, color="#D92D20", edgecolor="black", zorder=3, label="Ego ingredient")
    axis.scatter(
        [positions[alter][0] for alter, _ in top_alters],
        [positions[alter][1] for alter, _ in top_alters],
        s=520,
        color="#84CAFF",
        edgecolor="#175CD3",
        zorder=3,
        label="Co-ingredient alters",
    )
    axis.annotate(
        f"{ego_ingredient}\n{ego_recipe_count:,} recipes",
        (0, 0),
        xytext=(0, -48),
        textcoords="offset points",
        ha="center",
        va="top",
        fontsize=9,
        fontweight="bold",
    )
    for alter, shared_recipe_count in top_alters:
        axis.annotate(
            f"{alter}\n{shared_recipe_count:,}",
            positions[alter],
            xytext=(0, 11),
            textcoords="offset points",
            ha="center",
            fontsize=8,
        )
    axis.set_title(f"{ego_ingredient.title()} - top co-ingredients")
    axis.set_aspect("equal")
    axis.set_xlim(-1.6, 1.6)
    axis.set_ylim(-1.45, 1.45)
    axis.axis("off")

fig.suptitle("Food.com ingredient ego networks (all recipes)")
fig.tight_layout()
ingredient_egonets_figure_path = figures_dir / "prominent_ingredient_egonets.png"
fig.savefig(ingredient_egonets_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

ingredient_ego_summary = pd.DataFrame(
    [
        {"rank": rank, "ingredient": ingredient, "recipes_containing_ingredient": count}
        for rank, (ingredient, count) in enumerate(prominent_ingredients, start=1)
    ]
)
ingredient_node_data = pd.DataFrame(ingredient_node_rows)
ingredient_edge_data = pd.DataFrame(ingredient_edge_rows)
ingredient_ego_summary.to_csv(figures_dir / "prominent_ingredients_by_recipe_count.csv", index=False)
ingredient_node_data.to_csv(figures_dir / "prominent_ingredient_egonet_nodes.csv", index=False)
ingredient_edge_data.to_csv(figures_dir / "prominent_ingredient_egonet_edges.csv", index=False)
print(f"Recipes analyzed: {len(ingredient_data):,}")
print(f"Distinct normalized ingredients: {len(ingredient_frequency):,}")
display(ingredient_ego_summary)
display(ingredient_edge_data)
print(f"Saved ingredient ego-network figure: {ingredient_egonets_figure_path}")

## Ingredient prevalence over time

For each recipe submission year, calculate the percentage of all recipes submitted that year containing salt, butter, or sugar. The denominator is all recipes with a valid submission date in that year. Ingredient strings are normalized in the same way as the ego-network analysis; each recipe contributes at most once per ingredient. The vertical marker identifies the first year with fewer than 1,000 submitted recipes, where percentages may be more volatile because the yearly sample is small.

In [ ]:
figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
ingredient_trend_data = pd.read_csv(
    ingredient_recipes_path,
    usecols=["id", "submitted", "ingredients"],
    dtype={"id": "string", "submitted": "string", "ingredients": "string"},
)
if ingredient_trend_data["id"].isna().any() or ingredient_trend_data["id"].duplicated().any():
    raise ValueError("Recipe IDs must be present and unique for yearly ingredient prevalence.")
ingredient_trend_data["submission_year"] = pd.to_datetime(
    ingredient_trend_data["submitted"], errors="coerce"
).dt.year
invalid_submission_years = int(ingredient_trend_data["submission_year"].isna().sum())
if invalid_submission_years:
    raise ValueError(
        f"Cannot calculate yearly ingredient prevalence: {invalid_submission_years} recipes "
        "have invalid submission dates."
    )
if ingredient_trend_data["ingredients"].isna().any():
    raise ValueError("Missing ingredient lists prevent complete yearly prevalence estimates.")

ingredient_targets = [ingredient for ingredient, _ in prominent_ingredients]
yearly_recipe_totals = Counter()
yearly_ingredient_recipe_counts = Counter()
ingredient_trend_parse_errors = []
for submission_year, recipe_id, raw_ingredients in ingredient_trend_data[
    ["submission_year", "id", "ingredients"]
].itertuples(index=False, name=None):
    try:
        parsed_ingredients = ast.literal_eval(raw_ingredients)
    except (SyntaxError, ValueError) as error:
        ingredient_trend_parse_errors.append((recipe_id, str(error)))
        continue
    if not isinstance(parsed_ingredients, list) or not all(
        isinstance(ingredient, str) for ingredient in parsed_ingredients
    ):
        ingredient_trend_parse_errors.append((recipe_id, "ingredient value is not a list of strings"))
        continue
    normalized_ingredients = {
        " ".join(ingredient.lower().split())
        for ingredient in parsed_ingredients
        if ingredient.strip()
    }
    year = int(submission_year)
    yearly_recipe_totals[year] += 1
    for ingredient in ingredient_targets:
        if ingredient in normalized_ingredients:
            yearly_ingredient_recipe_counts[(year, ingredient)] += 1

if ingredient_trend_parse_errors:
    raise ValueError(
        f"Could not parse ingredient lists for {len(ingredient_trend_parse_errors)} recipes; "
        f"first examples: {ingredient_trend_parse_errors[:5]}"
    )

trend_years = range(min(yearly_recipe_totals), max(yearly_recipe_totals) + 1)
ingredient_prevalence_rows = []
for year in trend_years:
    total_recipes = yearly_recipe_totals[year]
    row = {"year": year, "recipes_submitted": total_recipes}
    for ingredient in ingredient_targets:
        recipe_count = yearly_ingredient_recipe_counts[(year, ingredient)]
        row[f"{ingredient}_recipe_count"] = recipe_count
        row[f"{ingredient}_share_pct"] = 100 * recipe_count / total_recipes if total_recipes else 0.0
    ingredient_prevalence_rows.append(row)
ingredient_prevalence = pd.DataFrame(ingredient_prevalence_rows)

fig, axis = plt.subplots(figsize=(12, 6))
for ingredient in ingredient_targets:
    axis.plot(
        ingredient_prevalence["year"],
        ingredient_prevalence[f"{ingredient}_share_pct"],
        marker="o",
        label=ingredient.title(),
    )
axis.set_title("Share of Food.com recipes containing prominent ingredients")
axis.set_xlabel("Recipe submission year")
axis.set_ylabel("Recipes containing ingredient (%)")
axis.grid(True, alpha=0.3)
axis.legend(title="Ingredient")
low_volume_years = ingredient_prevalence.loc[
    ingredient_prevalence["recipes_submitted"] < 1000, "year"
]
if not low_volume_years.empty:
    low_volume_start_year = int(low_volume_years.min())
    axis.axvline(
        low_volume_start_year,
        color="black",
        linestyle="--",
        alpha=0.7,
        label="First year with <1,000 submissions",
    )
    axis.axvspan(low_volume_start_year, max(ingredient_prevalence["year"]), color="gray", alpha=0.08)
    axis.legend(title="Ingredient", loc="best")
fig.tight_layout()
ingredient_prevalence_figure_path = figures_dir / "prominent_ingredient_prevalence_over_time.png"
fig.savefig(ingredient_prevalence_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

ingredient_prevalence_csv_path = figures_dir / "prominent_ingredient_prevalence_over_time.csv"
ingredient_prevalence.to_csv(ingredient_prevalence_csv_path, index=False)
print(f"Recipe submission years: {ingredient_prevalence['year'].min()}-{ingredient_prevalence['year'].max()}")
print(f"Invalid submission dates excluded: {invalid_submission_years}")
display(ingredient_prevalence.round(2))
print(f"Saved ingredient prevalence figure: {ingredient_prevalence_figure_path}")
print(f"Saved ingredient prevalence data: {ingredient_prevalence_csv_path}")

### Interpretation notes

The percentages describe the prevalence of an ingredient in submitted recipes, not the share of all Food.com recipes ever published and not the amount of the ingredient used. From 2001 through 2014, salt's share mostly stays between about 36% and 40%. Butter declines from 24.5% in 2001 to 18.4% in 2013, and sugar declines from 21.9% to 16.4% over the same period. The dashed marker identifies 2015 as the first year with fewer than 1,000 recipe submissions; only 189-306 recipes were submitted per year in 2015-2018, so those later percentages fluctuate more and should be treated cautiously. Ingredient names are user-entered strings, so spelling variants and synonyms may be counted separately.

## Ingredients with the largest relative change

To identify emerging or declining ingredients, rank normalized ingredient strings by the slope of their annual recipe share over 2000-2014. Also compare pooled recipe shares between an early baseline (2000-2005) and a later period (2010-2014); relative change is later share divided by early share. Pooled shares use each period's total recipe submissions as denominator. The trend window stops in 2014 because only 189-306 recipes per year appear in 2015-2018, making end-of-file percentages highly volatile. To avoid ranking trivial one-off strings, slope rankings require at least 30 recipe occurrences across 2000-2014 and presence in at least 3 years. Ratio rankings additionally require at least 5 early-period and 10 late-period recipes; ratios near this minimum remain sensitive to small baselines. Ingredient aliases are not merged, and slopes describe association over time rather than cause.

In [ ]:
import ast
import numpy as np
from collections import Counter, defaultdict

INGREDIENT_TREND_START_YEAR = 2000
INGREDIENT_TREND_END_YEAR = 2014
EARLY_INGREDIENT_YEARS = range(2000, 2006)
LATE_INGREDIENT_YEARS = range(2010, 2015)
MIN_TREND_RECIPE_OCCURRENCES = 30
MIN_TREND_ACTIVE_YEARS = 3
MIN_EARLY_RECIPE_OCCURRENCES_FOR_RATIO = 5
MIN_LATE_RECIPE_OCCURRENCES_FOR_RATIO = 10

ingredient_trend_counts = defaultdict(Counter)
ingredient_trend_totals = Counter()
ingredient_trend_parse_errors = []
for submission_year, recipe_id, raw_ingredients in ingredient_trend_data[
    ["submission_year", "id", "ingredients"]
].itertuples(index=False, name=None):
    year = int(submission_year)
    if not INGREDIENT_TREND_START_YEAR <= year <= INGREDIENT_TREND_END_YEAR:
        continue
    try:
        parsed_ingredients = ast.literal_eval(raw_ingredients)
    except (SyntaxError, ValueError) as error:
        ingredient_trend_parse_errors.append((recipe_id, str(error)))
        continue
    if not isinstance(parsed_ingredients, list) or not all(
        isinstance(ingredient, str) for ingredient in parsed_ingredients
    ):
        ingredient_trend_parse_errors.append((recipe_id, "ingredient value is not a list of strings"))
        continue
    normalized_ingredients = {
        " ".join(ingredient.lower().split())
        for ingredient in parsed_ingredients
        if ingredient.strip()
    }
    ingredient_trend_totals[year] += 1
    ingredient_trend_counts[year].update(normalized_ingredients)

if ingredient_trend_parse_errors:
    raise ValueError(
        f"Could not parse ingredient lists for {len(ingredient_trend_parse_errors)} recipes; "
        f"first examples: {ingredient_trend_parse_errors[:5]}"
    )
ingredient_trend_years = list(
    range(INGREDIENT_TREND_START_YEAR, INGREDIENT_TREND_END_YEAR + 1)
)
missing_trend_years = [year for year in ingredient_trend_years if not ingredient_trend_totals[year]]
if missing_trend_years:
    raise ValueError(f"No dated recipes were found in trend years: {missing_trend_years}")

ingredient_period_counts = Counter()
for year in ingredient_trend_years:
    ingredient_period_counts.update(ingredient_trend_counts[year])
early_total_recipes = sum(ingredient_trend_totals[year] for year in EARLY_INGREDIENT_YEARS)
late_total_recipes = sum(ingredient_trend_totals[year] for year in LATE_INGREDIENT_YEARS)
ingredient_ranking_rows = []
for ingredient, total_occurrences in ingredient_period_counts.items():
    annual_counts = [ingredient_trend_counts[year][ingredient] for year in ingredient_trend_years]
    active_years = sum(count > 0 for count in annual_counts)
    if total_occurrences < MIN_TREND_RECIPE_OCCURRENCES or active_years < MIN_TREND_ACTIVE_YEARS:
        continue
    annual_shares = [
        100 * ingredient_trend_counts[year][ingredient] / ingredient_trend_totals[year]
        for year in ingredient_trend_years
    ]
    slope_percentage_points_per_year = float(
        np.polyfit(ingredient_trend_years, annual_shares, 1)[0]
    )
    early_occurrences = sum(
        ingredient_trend_counts[year][ingredient] for year in EARLY_INGREDIENT_YEARS
    )
    late_occurrences = sum(
        ingredient_trend_counts[year][ingredient] for year in LATE_INGREDIENT_YEARS
    )
    early_share_pct = 100 * early_occurrences / early_total_recipes
    late_share_pct = 100 * late_occurrences / late_total_recipes
    ingredient_ranking_rows.append(
        {
            "ingredient": ingredient,
            "slope_percentage_points_per_year": slope_percentage_points_per_year,
            "early_share_pct_2000_2005": early_share_pct,
            "late_share_pct_2010_2014": late_share_pct,
            "relative_share_ratio_late_to_early": (
                late_share_pct / early_share_pct if early_share_pct > 0 else float("nan")
            ),
            "share_change_percentage_points": late_share_pct - early_share_pct,
            "early_recipe_count": early_occurrences,
            "late_recipe_count": late_occurrences,
            "total_recipe_count_2000_2014": total_occurrences,
            "active_years_2000_2014": active_years,
        }
    )
ingredient_change_rankings = pd.DataFrame(ingredient_ranking_rows)
if ingredient_change_rankings.empty:
    raise ValueError("No ingredients met the trend-ranking support thresholds.")
ingredient_change_rankings = ingredient_change_rankings.sort_values(
    ["slope_percentage_points_per_year", "ingredient"],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)
ingredient_change_rankings["slope_rank"] = ingredient_change_rankings.index + 1
ratio_eligible = ingredient_change_rankings.loc[
    (ingredient_change_rankings["early_recipe_count"] >= MIN_EARLY_RECIPE_OCCURRENCES_FOR_RATIO)
    & (ingredient_change_rankings["late_recipe_count"] >= MIN_LATE_RECIPE_OCCURRENCES_FOR_RATIO)
].sort_values(
    ["relative_share_ratio_late_to_early", "ingredient"],
    ascending=[False, True],
    kind="stable",
).copy()
ingredient_change_rankings["relative_change_rank"] = pd.NA
ingredient_change_rankings.loc[ratio_eligible.index, "relative_change_rank"] = range(
    1, len(ratio_eligible) + 1
)
display_columns = [
    "ingredient",
    "slope_percentage_points_per_year",
    "early_share_pct_2000_2005",
    "late_share_pct_2010_2014",
    "relative_share_ratio_late_to_early",
    "early_recipe_count",
    "late_recipe_count",
    "total_recipe_count_2000_2014",
]
top_slope_increases = ingredient_change_rankings.head(15)
top_slope_declines = ingredient_change_rankings.tail(15).sort_values(
    "slope_percentage_points_per_year", ascending=True
)
top_relative_changes = ratio_eligible.head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
increase_plot = top_slope_increases.sort_values("slope_percentage_points_per_year")
axes[0].barh(
    increase_plot["ingredient"],
    increase_plot["slope_percentage_points_per_year"],
    color="tab:green",
)
axes[0].set_title("Largest positive annual-share slopes")
axes[0].set_xlabel("Percentage points per year (2000-2014)")
axes[0].grid(True, axis="x", alpha=0.3)
ratio_plot = top_relative_changes.sort_values("relative_share_ratio_late_to_early")
axes[1].barh(
    ratio_plot["ingredient"],
    ratio_plot["relative_share_ratio_late_to_early"],
    color="tab:blue",
)
axes[1].set_title("Largest early-to-late share ratios")
axes[1].set_xlabel("2010-2014 share / 2000-2005 share")
axes[1].grid(True, axis="x", alpha=0.3)
fig.suptitle("Ingredients with the largest prevalence changes in submitted recipes")
fig.tight_layout()
ingredient_change_ranking_figure = figures_dir / "ingredient_relative_change_rankings.png"
fig.savefig(ingredient_change_ranking_figure, dpi=200, bbox_inches="tight")
plt.close(fig)

example_ingredients = ["quinoa", "kale", "sriracha sauce", "coconut oil"]
example_trend_rows = []
for year in ingredient_trend_years:
    row = {"year": year, "recipes_submitted": ingredient_trend_totals[year]}
    for ingredient in example_ingredients:
        count = ingredient_trend_counts[year][ingredient]
        row[f"{ingredient}_recipe_count"] = count
        row[f"{ingredient}_share_pct"] = 100 * count / ingredient_trend_totals[year]
    example_trend_rows.append(row)
example_ingredient_trends = pd.DataFrame(example_trend_rows)
fig, axis = plt.subplots(figsize=(12, 6))
for ingredient in example_ingredients:
    axis.plot(
        example_ingredient_trends["year"],
        example_ingredient_trends[f"{ingredient}_share_pct"],
        marker="o",
        label=ingredient.title(),
    )
axis.set_title("Recipe-share trends for selected emerging ingredients")
axis.set_xlabel("Recipe submission year")
axis.set_ylabel("Recipes containing ingredient (%)")
axis.legend(title="Ingredient")
axis.grid(True, alpha=0.3)
fig.tight_layout()
example_ingredient_figure = figures_dir / "selected_emerging_ingredient_trends.png"
fig.savefig(example_ingredient_figure, dpi=200, bbox_inches="tight")
plt.close(fig)

ingredient_change_ranking_path = figures_dir / "ingredient_relative_change_rankings.csv"
example_ingredient_trends_path = figures_dir / "selected_emerging_ingredient_trends.csv"
ingredient_change_rankings.to_csv(ingredient_change_ranking_path, index=False)
example_ingredient_trends.to_csv(example_ingredient_trends_path, index=False)
print("Top ingredients by positive slope:")
display(top_slope_increases[display_columns].round(3))
print("Ingredients with the most negative slopes:")
display(top_slope_declines[display_columns].round(3))
print("Top relative increases (minimum 5 early and 10 late recipe occurrences):")
display(top_relative_changes[display_columns].round(3))
print(f"Saved complete rankings: {ingredient_change_ranking_path}")
print(f"Saved ranking chart: {ingredient_change_ranking_figure}")
print(f"Saved selected ingredient trend chart: {example_ingredient_figure}")
print(f"Saved selected ingredient yearly data: {example_ingredient_trends_path}")

### Interpretation notes

The slope ranking emphasizes steady changes across 2000-2014, while the ratio ranking emphasizes proportional change between the chosen early and late windows. Large ratios can result from small early baselines, so read them with the listed early and late recipe counts and percentage-point change. Exact ingredient text is used after lowercasing and whitespace normalization; for example, the dataset labels the example condiment `sriracha sauce`. These tags are recipe ingredient strings and may include variants that are not combined.

In the early-versus-late ratio ranking, coconut oil is first (52.1x; 8 recipes in 2000-2005 versus 146 in 2010-2014), followed by smoked paprika (49.5x) and sriracha sauce (45.1x; 5 versus 79 recipes). Quinoa is 14.5x (37 versus 188 recipes), while the exact ingredient string `kale` is 5.3x (64 versus 118). These large ratios represent growth from low early shares; they should be read alongside raw counts and percentage-point changes. The example ingredients all increase in prevalence over much of 2000-2014, but coconut oil and sriracha sauce have much lower starting counts than quinoa or kale.

## Build yearly ego networks

For each year, find recipes reviewed by the ego and count how many distinct ones each other user also reviewed that year. This gives a yearly weighted ego network. The change measures compare each calendar year with the previous one; years with no ego activity remain in the series so gaps are visible.

In [3]:
first_year = int(events["year"].min())
last_year = int(events["year"].max())
metric_rows = []
edge_rows = []
previous_alters = set()

for year in range(first_year, last_year + 1):
    period = events.loc[events["year"] == year]
    ego_reviews = period.loc[period["user_id"] == ego_id]
    ego_recipes = set(ego_reviews["recipe_id"])

    shared_counts = (
        period.loc[period["recipe_id"].isin(ego_recipes)]
        .groupby("user_id")["recipe_id"]
        .nunique()
        .drop(index=ego_id, errors="ignore")
        .sort_values(ascending=False)
    )
    current_alters = set(shared_counts.index)
    retained_alters = current_alters & previous_alters

    metric_rows.append(
        {
            "year": year,
            "ego_recipes": len(ego_recipes),
            "alters": len(current_alters),
            "total_shared_recipes": int(shared_counts.sum()),
            "mean_shared_recipes": float(shared_counts.mean()) if len(shared_counts) else 0.0,
            "new_alters": len(current_alters - previous_alters),
            "retained_alters": len(retained_alters),
            "lost_alters": len(previous_alters - current_alters),
            "alter_retention": (
                len(retained_alters) / len(previous_alters)
                if previous_alters
                else float("nan")
            ),
        }
    )
    edge_rows.extend(
        {
            "year": year,
            "ego": ego_id,
            "alter": alter_id,
            "shared_recipes": int(shared_recipe_count),
        }
        for alter_id, shared_recipe_count in shared_counts.items()
    )
    previous_alters = current_alters

yearly_metrics = pd.DataFrame(metric_rows)
ego_edges = pd.DataFrame(
    edge_rows,
    columns=["year", "ego", "alter", "shared_recipes"],
)

display(yearly_metrics)

,year,ego_recipes,alters,total_shared_recipes,mean_shared_recipes,new_alters,retained_alters,lost_alters,alter_retention
0,2000,0,0,0,0.000000,0,0,0,NaN
1,2001,4,1,1,1.000000,1,0,0,NaN
2,2002,23,186,219,1.177419,186,0,1,0.000000
3,2003,37,138,157,1.137681,117,21,165,0.112903
4,2004,104,261,308,1.180077,234,27,111,0.195652
5,2005,50,170,184,1.082353,150,20,241,0.076628
6,2006,67,241,261,1.082988,222,19,151,0.111765
7,2007,53,402,446,1.109453,375,27,214,0.112033
8,2008,77,461,502,1.088937,423,38,364,0.094527
9,2009,105,451,477,1.057650,411,40,421,0.086768


In [ ]:
figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
plot_specs = [
    ("ego_recipes", "Distinct recipes reviewed by ego"),
    ("alters", "Co-review alters"),
    ("mean_shared_recipes", "Mean shared recipes per tie"),
    ("alter_retention", "Alter retention from previous year"),
]
for axis, (column, title) in zip(axes.flat, plot_specs):
    axis.plot(yearly_metrics["year"], yearly_metrics[column], marker="o")
    axis.set_title(title)
    axis.set_xlabel("Year")
    axis.grid(True, alpha=0.3)
axes.flat[3].set_ylim(0, 1)
fig.suptitle(f"Yearly ego-network change for Food.com user {ego_id}")
fig.tight_layout()
figure_path = figures_dir / "ego_network_over_time.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved figure to {figure_path}")

active_years = yearly_metrics.loc[yearly_metrics["ego_recipes"] > 0, "year"]
if active_years.empty:
    raise ValueError(f"Ego user {ego_id!r} has no valid activity in the selected date range.")
latest_active_year = int(active_years.max())
print(f"Strongest ego ties in {latest_active_year}:")
strongest_ties = (
    ego_edges.loc[ego_edges["year"] == latest_active_year]
    .sort_values("shared_recipes", ascending=False)
    .head(20)
)
display(strongest_ties)

metrics_path = figures_dir / "ego_yearly_metrics.csv"
edges_path = figures_dir / "ego_yearly_edges.csv"
strongest_ties_path = figures_dir / "strongest_ties_latest_year.csv"
yearly_metrics.to_csv(metrics_path, index=False)
ego_edges.to_csv(edges_path, index=False)
strongest_ties.to_csv(strongest_ties_path, index=False)
print(f"Saved yearly metrics to {metrics_path}")
print(f"Saved yearly ego edges to {edges_path}")
print(f"Saved strongest ties to {strongest_ties_path}")

## How to interpret the results

- **Alters** is the number of other users who reviewed at least one of the ego's recipes that year.
- **Shared recipes** measures tie strength as recipe overlap, not number of reviews, likes, or direct interactions between the two users.
- **New/lost/retained alters** and **alter retention** describe how stable this inferred neighborhood is between calendar years. Retention is undefined for the first year.
- These patterns can reflect changes in the ego's activity, site usage, or available data as well as changes in user overlap. They should not be described as changes in Twitter friendships or following relationships.

Use the plotted trends and the strongest-tie table to write up what changes over time for the selected ego. The figure and exported results tables are saved in the `figures` folder.

## Vegan and vegetarian recipe trends over time

This section uses the explicit `vegan` and `vegetarian` tags on Food.com recipes, rather than inferring dietary suitability from names or ingredients. It compares (1) the share of recipes submitted each year with those tags, (2) the share of reviews on tagged recipes, and (3) average and median review ratings by review year. 2018 is excluded throughout this section so the analysis ends in 2017. Vertical dashed lines mark the first year below the displayed low-volume thresholds (fewer than 1,000 recipe submissions or 25,000 reviews). These thresholds are visual caution flags, not formal statistical cutoffs. Vegan-tagged recipes are also vegetarian-tagged in this dataset, so vegan, vegetarian-only, and combined vegetarian-or-vegan categories are reported separately. Zero ratings are excluded from rating statistics because they are unrated records in this dataset. These measures describe Food.com content and activity; they do not identify reviewers' diets or prove a change in population prevalence.

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

dataset_path = Path(dataset_path)
recipes_path = dataset_path / "RAW_recipes.csv"
reviews_path = dataset_path / "RAW_interactions.csv"
if not recipes_path.is_file() or not reviews_path.is_file():
    raise FileNotFoundError(
        "Expected RAW_recipes.csv and RAW_interactions.csv in the KaggleHub dataset path: "
        f"{dataset_path}"
    )

recipe_data = pd.read_csv(
    recipes_path,
    usecols=["id", "submitted", "tags"],
    dtype={"id": "string", "submitted": "string", "tags": "string"},
)
if recipe_data["id"].duplicated().any():
    raise ValueError("RAW_recipes.csv contains duplicate recipe IDs; refusing ambiguous review matching.")
recipe_data["submitted_year"] = pd.to_datetime(
    recipe_data["submitted"], errors="coerce"
).dt.year
invalid_submission_dates = int(recipe_data["submitted_year"].isna().sum())

def parse_recipe_tags(value):
    if pd.isna(value):
        return set()
    parsed_tags = ast.literal_eval(value)
    if not isinstance(parsed_tags, list):
        raise ValueError("A recipe tags value was not a list.")
    return {str(tag).casefold() for tag in parsed_tags}

tag_sets = recipe_data["tags"].map(parse_recipe_tags)
recipe_data["is_vegan"] = tag_sets.map(lambda tags: "vegan" in tags)
recipe_data["is_vegetarian"] = tag_sets.map(lambda tags: "vegetarian" in tags)
recipe_data["is_vegetarian_or_vegan"] = recipe_data["is_vegetarian"] | recipe_data["is_vegan"]
recipe_data["is_vegetarian_only"] = recipe_data["is_vegetarian"] & ~recipe_data["is_vegan"]

valid_recipes = recipe_data.dropna(subset=["submitted_year"]).copy()
valid_recipes = valid_recipes.loc[valid_recipes["submitted_year"] < 2018].copy()
recipe_trends = (
    valid_recipes.groupby("submitted_year")
    .agg(
        recipes_submitted=("id", "nunique"),
        vegan_recipes=("is_vegan", "sum"),
        vegetarian_only_recipes=("is_vegetarian_only", "sum"),
        vegetarian_or_vegan_recipes=("is_vegetarian_or_vegan", "sum"),
    )
    .rename_axis("year")
)
for category in ["vegan", "vegetarian_only", "vegetarian_or_vegan"]:
    recipe_trends[f"{category}_recipe_share_pct"] = (
        100 * recipe_trends[f"{category}_recipes"] / recipe_trends["recipes_submitted"]
    )

interaction_data = pd.read_csv(
    reviews_path,
    usecols=["recipe_id", "date", "rating"],
    dtype={"recipe_id": "string", "date": "string"},
)
interaction_data["review_year"] = pd.to_datetime(
    interaction_data["date"], errors="coerce"
).dt.year
invalid_review_dates = int(interaction_data["review_year"].isna().sum())
valid_interactions = interaction_data.dropna(subset=["recipe_id", "review_year", "rating"]).copy()
valid_interactions["review_year"] = valid_interactions["review_year"].astype(int)
valid_interactions = valid_interactions.loc[valid_interactions["review_year"] < 2018].copy()
recipe_flags = recipe_data[
    ["id", "is_vegan", "is_vegetarian_only", "is_vegetarian_or_vegan"]
].rename(columns={"id": "recipe_id"})
matched_reviews = valid_interactions.merge(
    recipe_flags,
    on="recipe_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_recipe_reviews = int((matched_reviews["_merge"] != "both").sum())
classified_reviews = matched_reviews.loc[matched_reviews["_merge"] == "both"].copy()
review_trends = (
    classified_reviews.groupby("review_year")
    .agg(
        classified_reviews=("recipe_id", "size"),
        vegan_recipe_reviews=("is_vegan", "sum"),
        vegetarian_only_recipe_reviews=("is_vegetarian_only", "sum"),
        vegetarian_or_vegan_recipe_reviews=("is_vegetarian_or_vegan", "sum"),
    )
    .rename_axis("year")
)
for category in ["vegan", "vegetarian_only", "vegetarian_or_vegan"]:
    review_trends[f"{category}_review_share_pct"] = (
        100 * review_trends[f"{category}_recipe_reviews"] / review_trends["classified_reviews"]
    )

rated_reviews = classified_reviews.loc[classified_reviews["rating"] > 0].copy()
rating_trends = pd.DataFrame(index=sorted(rated_reviews["review_year"].unique()))
rating_trends.index.name = "year"
rating_category_flags = {
    "vegan": "is_vegan",
    "vegetarian_only": "is_vegetarian_only",
    "vegetarian_or_vegan": "is_vegetarian_or_vegan",
}
for category, flag_column in rating_category_flags.items():
    category_ratings = rated_reviews.loc[rated_reviews[flag_column]].groupby("review_year")["rating"]
    rating_trends[f"{category}_rating_count"] = category_ratings.count()
    rating_trends[f"{category}_rating_mean"] = category_ratings.mean()
    rating_trends[f"{category}_rating_median"] = category_ratings.median()

annual_diet_trends = recipe_trends.join(review_trends, how="outer").join(rating_trends, how="outer").sort_index()
print(f"Recipes with invalid submission dates excluded: {invalid_submission_dates}")
print(f"Reviews with invalid dates excluded: {invalid_review_dates}")
print(f"Reviews for recipe IDs not matched to RAW_recipes.csv: {unmatched_recipe_reviews}")
print(f"Zero-rated reviews excluded from rating summaries: {int((classified_reviews['rating'] == 0).sum())}")
display(annual_diet_trends.round(2))

figures_dir = Path("figures")
figures_dir.mkdir(parents=True, exist_ok=True)
LOW_RECIPE_COUNT = 1000
LOW_REVIEW_COUNT = 25000
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
recipe_series = [
    ("vegan_recipe_share_pct", "Vegan-tagged"),
    ("vegetarian_only_recipe_share_pct", "Vegetarian-only tagged"),
    ("vegetarian_or_vegan_recipe_share_pct", "Vegetarian or vegan tagged"),
]
review_series = [
    ("vegan_review_share_pct", "Reviews of vegan-tagged recipes"),
    ("vegetarian_only_review_share_pct", "Reviews of vegetarian-only recipes"),
    ("vegetarian_or_vegan_review_share_pct", "Reviews of vegetarian-or-vegan recipes"),
]
for column, label in recipe_series:
    axes[0].plot(annual_diet_trends.index, annual_diet_trends[column], marker="o", label=label)
axes[0].set_title("Share of recipes submitted by year")
axes[0].set_ylabel("Recipes (%)")
axes[0].set_xlabel("Submission year")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
low_recipe_years = recipe_trends.index[
    recipe_trends["recipes_submitted"] < LOW_RECIPE_COUNT
]
if len(low_recipe_years):
    axes[0].axvline(
        low_recipe_years.min() - 0.5,
        color="crimson",
        linestyle="--",
        linewidth=1.5,
        label=f"Low volume begins (<{LOW_RECIPE_COUNT:,} recipes/year)",
    )
    axes[0].legend()
for column, label in review_series:
    axes[1].plot(annual_diet_trends.index, annual_diet_trends[column], marker="o", label=label)
axes[1].set_title("Share of reviews by review year")
axes[1].set_ylabel("Reviews (%)")
axes[1].set_xlabel("Review year")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
low_review_years = review_trends.index[
    review_trends["classified_reviews"] < LOW_REVIEW_COUNT
]
if len(low_review_years):
    axes[1].axvline(
        low_review_years.min() - 0.5,
        color="crimson",
        linestyle="--",
        linewidth=1.5,
        label=f"Low volume begins (<{LOW_REVIEW_COUNT:,} reviews/year)",
    )
    axes[1].legend()
fig.suptitle("Vegan and vegetarian signals on Food.com (through 2017)")
fig.tight_layout()
diet_figure_path = figures_dir / "vegan_vegetarian_trends.png"
fig.savefig(diet_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

recipe_count_trends = recipe_trends[
    [
        "recipes_submitted",
        "vegan_recipes",
        "vegetarian_only_recipes",
        "vegetarian_or_vegan_recipes",
    ]
].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(
    recipe_count_trends.index,
    recipe_count_trends["recipes_submitted"],
    marker="o",
    color="gray",
)
axes[0].set_title("All recipes submitted")
axes[0].set_ylabel("Number of recipes")
axes[0].set_xlabel("Submission year")
axes[0].grid(True, alpha=0.3)
axes[1].plot(
    recipe_count_trends.index,
    recipe_count_trends["vegan_recipes"],
    marker="o",
    label="Vegan-tagged",
)
axes[1].plot(
    recipe_count_trends.index,
    recipe_count_trends["vegetarian_only_recipes"],
    marker="o",
    label="Vegetarian-only tagged",
)
axes[1].plot(
    recipe_count_trends.index,
    recipe_count_trends["vegetarian_or_vegan_recipes"],
    marker="o",
    label="Vegetarian or vegan tagged (combined)",
)
axes[1].set_title("Vegetarian and vegan recipes submitted")
axes[1].set_ylabel("Number of recipes")
axes[1].set_xlabel("Submission year")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
low_recipe_count_years = recipe_count_trends.index[
    recipe_count_trends["recipes_submitted"] < LOW_RECIPE_COUNT
]
if len(low_recipe_count_years):
    for axis in axes:
        axis.axvline(
            low_recipe_count_years.min() - 0.5,
            color="crimson",
            linestyle="--",
            linewidth=1.5,
            label="Low total volume begins (<1,000 recipes/year)",
        )
    axes[0].legend()
    axes[1].legend()
fig.suptitle("Actual number of Food.com recipe submissions (through 2017)")
fig.tight_layout()
recipe_count_figure_path = figures_dir / "vegan_vegetarian_recipe_counts.png"
fig.savefig(recipe_count_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)
display(recipe_count_trends.rename_axis("year").astype(int))
recipe_count_summary_path = figures_dir / "vegan_vegetarian_recipe_counts.csv"
recipe_count_trends.rename_axis("year").to_csv(recipe_count_summary_path)
print(f"Saved actual-count chart to {recipe_count_figure_path}")
print(f"Saved actual-count table to {recipe_count_summary_path}")

recipe_vegan_peak_year = int(recipe_trends["vegan_recipe_share_pct"].idxmax())
recipe_vegetarian_peak_year = int(recipe_trends["vegetarian_or_vegan_recipe_share_pct"].idxmax())
review_vegan_peak_year = int(review_trends["vegan_review_share_pct"].idxmax())
review_vegetarian_peak_year = int(review_trends["vegetarian_or_vegan_review_share_pct"].idxmax())
last_recipe_year = int(recipe_trends.index.max())
last_review_year = int(review_trends.index.max())
print(
    f"Vegan-tagged recipe share peaked in {recipe_vegan_peak_year} "
    f"({recipe_trends.loc[recipe_vegan_peak_year, 'vegan_recipe_share_pct']:.2f}%)."
)
print(
    f"Combined vegetarian-or-vegan recipe share peaked in {recipe_vegetarian_peak_year} "
    f"({recipe_trends.loc[recipe_vegetarian_peak_year, 'vegetarian_or_vegan_recipe_share_pct']:.2f}%)."
)
print(
    f"Vegan-tagged review share peaked in {review_vegan_peak_year} "
    f"({review_trends.loc[review_vegan_peak_year, 'vegan_review_share_pct']:.2f}%)."
)
print(
    f"Combined vegetarian-or-vegan review share peaked in {review_vegetarian_peak_year} "
    f"({review_trends.loc[review_vegetarian_peak_year, 'vegetarian_or_vegan_review_share_pct']:.2f}%)."
)
print(f"Analysis cutoff: 2017 (2018 excluded). Last recipe submission year included: {last_recipe_year}.")
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True)
for category, label in [
    ("vegan", "Vegan-tagged"),
    ("vegetarian_only", "Vegetarian-only"),
    ("vegetarian_or_vegan", "Vegetarian or vegan"),
]:
    axes[0].plot(
        rating_trends.index,
        rating_trends[f"{category}_rating_mean"],
        marker="o",
        label=label,
    )
    axes[1].plot(
        rating_trends.index,
        rating_trends[f"{category}_rating_median"],
        marker="o",
        label=label,
    )
axes[0].set_title("Mean review rating")
axes[1].set_title("Median review rating")
for axis in axes:
    axis.set_xlabel("Review year")
    axis.set_ylabel("Rating (1-5; zero excluded)")
    axis.set_ylim(1, 5.1)
    axis.grid(True, alpha=0.3)
    axis.legend()
fig.suptitle("Ratings of vegan and vegetarian recipes (through 2017)")
fig.tight_layout()
rating_figure_path = figures_dir / "vegan_vegetarian_ratings.png"
fig.savefig(rating_figure_path, dpi=200, bbox_inches="tight")
plt.close(fig)

rating_summary_columns = [
    column for column in rating_trends.columns
    if column.endswith("_rating_count") or column.endswith("_rating_mean") or column.endswith("_rating_median")
]
display(rating_trends[rating_summary_columns].round(2))
diet_summary_path = figures_dir / "vegan_vegetarian_annual_summary.csv"
rating_summary_path = figures_dir / "vegan_vegetarian_annual_ratings.csv"
annual_diet_trends.to_csv(diet_summary_path, index_label="year")
rating_trends.to_csv(rating_summary_path, index_label="year")
print(f"Saved figure to {diet_figure_path}")
print(f"Saved ratings figure to {rating_figure_path}")
print(f"Saved annual summary to {diet_summary_path}")
print(f"Saved annual ratings to {rating_summary_path}")

### Interpretation notes

The analysis excludes 2018 and ends in 2017. In the actual-count chart, vegan-tagged recipes are a subset of vegetarian-or-vegan recipes; vegetarian-only and combined counts are shown separately so counts are not mistaken for disjoint groups. The dashed line marks the first year below 1,000 total recipe submissions; it is a heuristic warning rather than a guarantee that earlier years are unbiased. In the review chart, the threshold is first crossed in 2015 and the annual count rises back above it in 2017. Review ratings are calculated from interactions with ratings greater than zero; the yearly count columns show the sample size behind each mean and median. Median ratings can remain at 5 when most reviews are 5-star, even as the mean changes. The review-share and rating measures describe Food.com activity; they do not identify individual users' diets. User-applied recipe tags may be incomplete or inaccurate.